In [1]:
import numpy as np
from scipy.linalg import expm
from scipy.integrate import simpson
from pennylane.math import partial_trace

sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)


def kron_n(*args):
	result = args[0]
	for a in args[1:]:
		result = np.kron(result, a)
	return result


def XY(n, gamma, h):
	dim = 2 ** n
	H = np.zeros((dim, dim), dtype=complex)

	# Nearest-neighbor XX/YY interactions
	for i in range(n - 1):
		left = np.eye(2 ** i, dtype=complex)
		right = np.eye(2 ** (n - i - 2), dtype=complex)
		term_xx = ((1 + gamma) / 2) * kron_n(left, sx, sx, right)
		term_yy = ((1 - gamma) / 2) * kron_n(left, sy, sy, right)
		H -= term_xx + term_yy

	# Longitudinal field terms
	for i in range(n):
		left = np.eye(2 ** i, dtype=complex)
		right = np.eye(2 ** (n - i - 1), dtype=complex)
		term_z = kron_n(left, sz, right)
		H -= h * term_z

	return H


# Gibbs state calculation
def gibbs(H, beta):
	eig = np.linalg.eigvalsh(H)
	Z = np.sum(np.exp(-beta * eig))
	return expm(-beta * H) / Z


# Time evolution operator
def U(H, t):
	return expm(-1j * H * t)


# Paired density matrices from spherical coordinates
def rho_pair(theta, phi):
	s1 = np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)], dtype=complex)
	s2 = np.array([np.sin(theta / 2), -np.exp(1j * phi) * np.cos(theta / 2)], dtype=complex)

	rho1 = np.outer(s1, s1.conj())
	rho2 = np.outer(s2, s2.conj())
	return rho1, rho2


# Trace distance calculation
def trace_distance(rho, sigma):
	diff = rho - sigma
	eig = np.linalg.eigvals(diff)
	return np.sum(np.abs(eig)) / 2


def build_hamiltonian(n, gamma, h, h0, J0):
	# Environment Hamiltonian (n-1 spins)
	Henv = XY(n - 1, gamma, h)

	# System Hamiltonian (first spin)
	Hsys = h0 * np.kron(sz, np.eye(2 ** (n - 1), dtype=complex))

	# Interaction Hamiltonian (first two spins)
	interaction_term = ((1 + gamma) / 2 * np.kron(sx, sx) +
	                    (1 - gamma) / 2 * np.kron(sy, sy))
	Hint = J0 * np.kron(interaction_term, np.eye(2 ** (n - 2), dtype=complex))

	# Full Hamiltonian
	Htotal = (Hsys +
	          np.kron(np.eye(2, dtype=complex), Henv) +
	          Hint)

	return Htotal, Henv

In [26]:
n = 4
gamma = 1
h = 3
h0 = h
J0 = 1

H_total, Henv = build_hamiltonian(n, gamma, h, h0, J0)
rho1, rho2 = rho_pair(np.pi / 2, 0)
tstep = 0.001
beta_list = [0, 0.2, 0.5, 0.8, 1., 1.2, 2., 3., 4., 5., 10.]
N_list = []

for beta in beta_list:
	rho_env = gibbs(Henv, beta)
	rho1_list = []
	rho2_list = []
	for t in np.arange(0, 2 / 3 * n, tstep):
		V = U(H_total, t)
		rho1_list.append(partial_trace(V @ np.kron(rho1, rho_env) @ V.T.conj(), indices=range(1, n)))
		rho2_list.append(partial_trace(V @ np.kron(rho2, rho_env) @ V.T.conj(), indices=range(1, n)))

	trace_distance_list = [trace_distance(rho, sigma) for rho, sigma in zip(rho1_list, rho2_list)]
	D_trace_distance_list = np.gradient(trace_distance_list, tstep, edge_order=2)
	D_trace_distance_list_clipped = np.clip(D_trace_distance_list, 0, np.inf)
	N = simpson(D_trace_distance_list_clipped, dx=tstep)
	print(N)
	N_list.append(N)

0.10499965419384981
0.10499965419384813
0.1049996541938472
0.10499965419384559
0.10499965419384613
0.10499965419384737
0.10499965419384724
0.10499965419384273
0.10499965419384752
0.10499965419384669
0.10499965419384752
